In [1]:
from model import GPT
import torch
import tiktoken

/workspace/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True


Total params: 123653376


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8930.47it/s]


didn't crash


In [2]:
model = GPT.from_pretrained("gpt2")
model.eval()
model.to("cuda")

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
Total params: 123653376


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6758.75it/s]


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [11]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)
num_sentences = 5
tokens = tokens.unsqueeze(0).repeat(num_sentences, 1)
print(tokens.shape)
x = tokens.to("cuda")

max_length = 30

while x.size(1) < max_length:
    with torch.no_grad():
        logits = model(x)
        logits = logits[:, -1, :]
        # print(logits.shape)
        probs = torch.nn.functional.softmax(logits, dim=-1)

        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        xcol = torch.gather(topk_indices, -1, ix)
        x = torch.cat([x, xcol], dim=1)

for i in range(num_sentences):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

torch.Size([5, 8])
> Hello, I'm a language model, not a program.

So this morning I started studying for the interview in the lab. This was not
> Hello, I'm a language model, and one of the main things that bothers me when they create languages is how easy it becomes to create something that
> Hello, I'm a language model, and I wrote it off on the grounds that a language model would make me more fluent. But I'm not
> Hello, I'm a language model, I really like languages. I like languages because like, they're good. And the way we talk about languages
> Hello, I'm a language model, a language model I'm using for data modelling. All I did was test the results and then I wrote some


In [10]:
from transformers import pipeline, set_seed
gen = pipeline('text-generation', model='gpt2')
set_seed(42)
gen("Hello, I'm a language model,", max_length=30, num_return_sequences=5)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7400.54it/s]


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hello, I\'m a language model, so you can\'t just use the same data model and just use different languages. There\'s a lot of overlap between languages because there are so many different languages.\n\nBut in a lot of cases, I have a lot of different languages. There is a lot of confusion about what\'s right. I sometimes have to put a lot of stuff in the right order, but then it\'s not so clear what it is. Sometimes there are multiple languages that are using different languages. So I have a lot of confusion.\n\nSo, for example, if you\'re doing a lot of cross-platform development, where you\'re developing for a large cross-platform platform, you\'re using a lot of different languages. And you may not be able to understand the language. But if you do a lot of cross-platform development, you can understand the language.\n\nAnd so there\'s a lot of confusion.\n\nWhen I talk to people about languages, I\'m kind of using the same language, but it\'s different.\n\nSo, I 